# Customer Churn Prediction — PySpark MLlib

**End-to-end reproducible workflow** | IBM Telco Customer Churn | Classification

This notebook complements the scikit-learn implementation. It uses a **stratified train / validation / test split**, fits preprocessing on training data only, selects a model and classification threshold on validation data, and evaluates the final result on an untouched test set. **Metrics must be computed by running the notebook**; no results are assumed.

> For an apples-to-apples comparison, both notebooks must use the same rows, features, split, model settings and selection protocol. This notebook uses a separate validation set, so its scores are **not directly comparable** to the previously reported scikit-learn test scores.

## 0 — Setup
Run in Google Colab. PySpark installation may take a few minutes.

In [ ]:
%pip -q install pyspark scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

SEED = 42
spark = SparkSession.builder.appName("CustomerChurnMLlib").getOrCreate()
print("Spark version:", spark.version)

## 1 — Load and validate source data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
local_path = Path("data/Telco-Customer-Churn.csv")
pdf = pd.read_csv(local_path if local_path.exists() else DATA_URL)
assert {"customerID", "TotalCharges", "tenure", "Churn"}.issubset(pdf.columns)
assert pdf["customerID"].is_unique
assert pdf["Churn"].isin(["Yes", "No"]).all()
print("Rows and columns:", pdf.shape)
display(pdf.head(3))

## 2 — Documented data cleaning
Blank TotalCharges values are assumed to represent zero accumulated charges **only when tenure is zero**. We retain an indicator of original missingness. This is a domain assumption, not verified billing information.

In [ ]:
pdf["TotalCharges"] = pd.to_numeric(pdf["TotalCharges"], errors="coerce")
pdf["TotalCharges_missing"] = pdf["TotalCharges"].isna().astype(int)
missing = pdf["TotalCharges"].isna()
assert pdf.loc[missing, "tenure"].eq(0).all(), "Revisit missing-value policy"
pdf.loc[missing, "TotalCharges"] = 0.0
pdf["label"] = (pdf["Churn"] == "Yes").astype(float)
print("Original missing TotalCharges:", missing.sum())
print("Remaining nulls:", pdf.isna().sum().sum())

## 3 — Stratified splits
We use **64% train / 16% validation / 20% test** with the same random seed. Validation is used for model and threshold selection; test is used once for final reporting.

In [ ]:
train_val, test_pd = train_test_split(pdf, test_size=0.20, stratify=pdf["label"], random_state=SEED)
train_pd, val_pd = train_test_split(train_val, test_size=0.20, stratify=train_val["label"], random_state=SEED)
print({"train": len(train_pd), "validation": len(val_pd), "test": len(test_pd)})
print({"train_rate": train_pd.label.mean(), "val_rate": val_pd.label.mean(), "test_rate": test_pd.label.mean()})
train = spark.createDataFrame(train_pd)
val = spark.createDataFrame(val_pd)
test = spark.createDataFrame(test_pd)

## 4 — Feature engineering pipeline
`customerID`, `Churn` and `label` are excluded from model features. We fit categorical encoders on training data only. Numerical features are assembled and standardized for Logistic Regression; the same transformed feature vector is used for Random Forest.

In [ ]:
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "TotalCharges_missing"]
categorical_cols = [c for c in pdf.columns if c not in numeric_cols + ["customerID", "Churn", "label"]]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols], handleInvalid="keep")
num_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="numeric_features", handleInvalid="error")
scaler = StandardScaler(inputCol="numeric_features", outputCol="scaled_numeric", withStd=True, withMean=False)
feature_assembler = VectorAssembler(inputCols=["scaled_numeric"] + [f"{c}_ohe" for c in categorical_cols], outputCol="features")
preprocess = indexers + [encoder, num_assembler, scaler, feature_assembler]
print("Numerical:", numeric_cols)
print("Categorical:", categorical_cols)

## 5 — Train candidate models

In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.01),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=7, seed=SEED)
}
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
trained = {}
rows = []
for name, classifier in candidates.items():
    fitted = Pipeline(stages=preprocess + [classifier]).fit(train)
    pred_val = fitted.transform(val)
    auc = evaluator.evaluate(pred_val)
    rows.append({"model": name, "validation_roc_auc": auc})
    trained[name] = fitted
results = pd.DataFrame(rows).sort_values("validation_roc_auc", ascending=False)
display(results)
best_name = results.iloc[0]["model"]
model = trained[best_name]
print("Selected on validation:", best_name)

## 6 — Tune the decision threshold on validation only
For a churn-retention use case, we can choose the threshold that maximizes F1 on validation. Business cost-sensitive thresholding would be preferable if outreach costs and missed churn costs were known.

In [ ]:
def predictions_to_pandas(spark_df):
    return spark_df.select("label", "probability").toPandas().assign(
        score=lambda d: d["probability"].map(lambda v: float(v[1]))
    )

val_scores = predictions_to_pandas(model.transform(val))
threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.05):
    predicted = (val_scores["score"] >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(val_scores["label"], predicted, zero_division=0),
        "recall": recall_score(val_scores["label"], predicted, zero_division=0),
        "f1": f1_score(val_scores["label"], predicted, zero_division=0)
    })
threshold_table = pd.DataFrame(threshold_rows).sort_values(["f1", "threshold"], ascending=[False, True])
display(threshold_table.round(3))
selected_threshold = float(threshold_table.iloc[0]["threshold"])
print("Threshold chosen on validation:", selected_threshold)

## 7 — Final test evaluation (do not tune after viewing these results)

In [ ]:
test_scores = predictions_to_pandas(model.transform(test))
y_true = test_scores["label"].astype(int)
y_score = test_scores["score"]
y_hat = (y_score >= selected_threshold).astype(int)
print("Selected model:", best_name)
print("Selected threshold:", selected_threshold)
print("Test ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))
print("Test Average Precision:", round(average_precision_score(y_true, y_score), 4))
print("Test confusion matrix:\n", confusion_matrix(y_true, y_hat))
print(classification_report(y_true, y_hat, target_names=["No churn", "Churn"], zero_division=0))

## 8 — Save model and document results
Saving a Spark PipelineModel writes a directory, not a single `.joblib` file. Record the dataset source, seed, versions, model and threshold in the README. Avoid committing large model artifacts by default.

In [ ]:
from pathlib import Path
output = "models/pyspark_customer_churn_pipeline"
Path("models").mkdir(exist_ok=True)
model.write().overwrite().save(output)
print("Saved Spark pipeline to:", output)
print("Spark version:", spark.version)
print("Selected threshold:", selected_threshold)

## Interpretation and next steps

- Compare the PySpark and scikit-learn approaches **qualitatively** first; their evaluation protocols differ.
- To compare numerical scores fairly, align the exact split, feature transformations, model hyperparameters and threshold policy.
- Report confusion matrix, class-1 recall and precision alongside ROC-AUC.
- Do not interpret predictive associations as causal effects.
- Keep the original notebook in an `archive/` folder when reorganizing GitHub.
